# Free Video GPU — Aura Marketing AI (learning project)

This notebook is **tier 2 and tier 3** of the app's free video-generation pipeline (see `backend/generator.py`):

- **Part A — Batch restock**: generates a handful of real AI video clips for free and fills `backend/generated_library/`, which your backend prefers over generic stock footage.
- **Part B — Live endpoint**: launches a small Gradio app with `share=True` (Gradio's own official temporary public-link feature) so your running backend can call your *own dedicated* Kaggle/Colab GPU on demand, ahead of the crowded public Hugging Face Space.

**Important, read before running:** this notebook exposes its Gradio demo using Gradio's built-in `share=True` link — the same mechanism people use to demo a Colab project publicly. It does **not** use ngrok or a raw reverse-proxy tunnel around the notebook UI, which is the specific pattern Google's Colab free-tier terms prohibit ("bypassing the notebook interface to interact mainly through another web interface"). Even so: this is still a *free, best-effort* GPU session, not a production server —
- The link is temporary and dies when the notebook session ends or disconnects.
- Kaggle gives you a dedicated 30 GPU-hours/week (T4, 16GB VRAM) on your own private session — no queue with the public internet.
- Colab's free GPU allocation is usage-based and not a published fixed quota — treat it as a bonus, not a guarantee.
- Never leave Part B running unattended for long periods just to keep a link alive — start it when you want to generate, then stop it.

## Setup
- **Kaggle**: Notebook Settings → Accelerator → GPU T4 x2, Internet → On.
- **Colab**: Runtime → Change runtime type → T4 GPU.


In [ ]:
# One-time setup. Kaggle/Colab usually already have a CUDA-enabled torch;
# this just adds the pieces that aren't preinstalled.
!pip install -q diffusers transformers accelerate imageio imageio-ffmpeg gradio

In [ ]:
import os
import json
import datetime
import torch
from diffusers import AutoModel, WanPipeline
from diffusers.utils import export_to_video

# Wan2.1's smallest text-to-video model: ~8.2GB VRAM, comfortably fits a
# free T4's 16GB. Swap for "Wan-AI/Wan2.1-T2V-14B-Diffusers" (needs
# offloading tricks, see the diffusers docs) if you have a bigger GPU.
MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"

print("Loading model — first run downloads ~3GB of weights, a few minutes...")
vae = AutoModel.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(MODEL_ID, vae=vae, torch_dtype=torch.bfloat16)
pipe.to("cuda")
print("Model loaded.")

In [ ]:
NEGATIVE_PROMPT = "blurry, low quality, static, worst quality, distorted, watermark"

def generate_clip(prompt: str, size: str = "832*480", num_frames: int = 49, fps: int = 16, out_path: str = "output.mp4") -> str:
    """
    Shared by both Part A (batch) and Part B (live endpoint). `size` matches
    the "W*H" convention backend/generator.py already sends. num_frames must
    be 4k+1 (Wan's requirement) — 49 is a fast ~3s clip; raise it for longer,
    slower renders.
    """
    width, height = (int(x) for x in size.split("*"))
    output = pipe(
        prompt=prompt,
        negative_prompt=NEGATIVE_PROMPT,
        num_frames=num_frames,
        guidance_scale=5.0,
        height=height,
        width=width,
    ).frames[0]
    export_to_video(output, out_path, fps=fps)
    return out_path

## Part A — Batch restock (fills `backend/generated_library/`)

Generates one clip per category your app already recognizes by keyword (see `SANDBOX_FEEDS` in `backend/generator.py`), writes a matching `manifest.json`, and zips the result for download. Run this whenever you want to refresh the library — it doesn't need Part B running.

In [ ]:
CATEGORY_PROMPTS = {
    "cyberpunk": {
        "prompt": "cinematic aerial shot flying over a futuristic neon-lit cyberpunk city at night, flying cars, rain-slicked streets, 8k",
        "tags": ["cyberpunk", "neon", "futuristic", "sci-fi", "city"],
    },
    "subway": {
        "prompt": "a sleek futuristic subway station with glowing neon lights, cinematic wide shot, 8k",
        "tags": ["subway", "train", "station"],
    },
    "nature": {
        "prompt": "a peaceful forest stream flowing over rocks in warm sunlight, cinematic nature footage, 8k",
        "tags": ["nature", "forest", "tree", "river", "water", "landscape"],
    },
    "abstract": {
        "prompt": "abstract digital animation of glowing green particles flowing through dark space, cinematic, 8k",
        "tags": ["abstract"],
    },
    "space": {
        "prompt": "a slow cinematic pan across stars and galaxies in deep space, cosmic nebula, 8k",
        "tags": ["space", "galaxy", "stars", "cosmos", "planet"],
    },
    "technology": {
        "prompt": "a glowing circuit board with pulsing lights of data flowing through it, macro cinematic shot, 8k",
        "tags": ["tech", "circuit", "code", "ai", "data", "binary"],
    },
    "ocean": {
        "prompt": "cinematic slow motion ocean waves crashing on a beach at golden hour, 8k",
        "tags": ["ocean", "sea", "wave", "beach"],
    },
    "anime": {
        "prompt": "flying through a colorful starry anime-style galaxy, vibrant cel-shaded art, cinematic, 8k",
        "tags": ["anime", "cartoon", "art", "manga"],
    },
}

OUT_DIR = "generated_library"
os.makedirs(OUT_DIR, exist_ok=True)

clips = []
for category, spec in CATEGORY_PROMPTS.items():
    filename = f"{category}_001.mp4"
    out_path = os.path.join(OUT_DIR, filename)
    print(f"Generating {category}...")
    generate_clip(spec["prompt"], size="832*480", out_path=out_path)
    clips.append({
        "file": filename,
        "tags": spec["tags"],
        "prompt": spec["prompt"],
        "source": "kaggle_or_colab_wan2.1_1.3b",
        "generated_at": datetime.datetime.utcnow().isoformat() + "Z",
    })
    print(f"  done -> {out_path}")

with open(os.path.join(OUT_DIR, "manifest.json"), "w") as f:
    json.dump({"clips": clips}, f, indent=2)

print("\nAll done. manifest.json written.")

In [ ]:
# Zip the library for easy download (Kaggle: see the notebook's Output
# tab; Colab: use the Files panel on the left, or files.download below).
import shutil
shutil.make_archive("generated_library", "zip", OUT_DIR)
print("Wrote generated_library.zip — download it, unzip, and copy the\n"
      "contents into backend/generated_library/ in your project, replacing\n"
      "the placeholder manifest.json there.")

try:
    from google.colab import files
    files.download("generated_library.zip")
except ImportError:
    pass  # not running on Colab; grab it from Kaggle's Output tab instead

## Part B — Live endpoint (optional, run while you want on-demand generation)

Launches a tiny Gradio app exposing one API endpoint (`/generate`) that mirrors what `backend/generator.py`'s `_private_gpu_worker` calls. Run this cell, copy the printed `https://....gradio.live` URL into your project's `.env` as `PRIVATE_GPU_ENDPOINT`, and restart your backend. Stop this cell (or let the session end) when you're done — the link stops working, and your app automatically falls back to the next free tier.

In [ ]:
import gradio as gr

def _live_generate(prompt: str, size: str):
    out_path = f"/tmp/live_{abs(hash(prompt)) % 10**8}.mp4"
    return generate_clip(prompt, size=size or "832*480", out_path=out_path)

demo = gr.Interface(
    fn=_live_generate,
    inputs=[gr.Textbox(label="prompt"), gr.Textbox(label="size", value="832*480")],
    outputs=gr.Video(label="generated_video"),
    api_name="generate",
    title="Aura free video GPU (private)",
)

demo.queue().launch(share=True)

print("\nCopy the https://....gradio.live URL printed above into your .env as:\n"
      "PRIVATE_GPU_ENDPOINT=https://<your-link>.gradio.live\n"
      "Then restart your backend. Stop this cell to shut the endpoint down.")